In [ ]:
# Cell 1: Load the SOWPODS word list from pre-extracted text file
import os, time
from collections import Counter

start = time.time()

# Find sowpods.txt in data directory
txt_path = None
for candidate in ['/workspace/data', 'data', '../data',
                  '/home/ddzhang/DSBench/colab_tasks/dsbench_da_00000009/environment/data']:
    p = os.path.join(candidate, 'sowpods.txt')
    if os.path.exists(p):
        txt_path = p
        break

if txt_path is None:
    raise FileNotFoundError("sowpods.txt not found in any data directory")

with open(txt_path, 'r') as f:
    all_words = [line.strip() for line in f if line.strip()]

print(f'Loaded {len(all_words)} words in {time.time()-start:.1f}s')

# Group words by length for fast lookup
words_by_length = {}
for w in all_words:
    n = len(w)
    if n not in words_by_length:
        words_by_length[n] = []
    words_by_length[n].append(w)

for n in [5, 6, 7]:
    print(f'  Words of length {n}: {len(words_by_length.get(n, []))}')

In [ ]:
# Cell 2: Define letter scores, board multipliers, and core functions

LETTER_POINTS = {
    'A': 1, 'B': 3, 'C': 3, 'D': 2, 'E': 1, 'F': 4, 'G': 2, 'H': 4,
    'I': 1, 'J': 8, 'K': 5, 'L': 1, 'M': 3, 'N': 1, 'O': 1, 'P': 3,
    'Q': 10, 'R': 1, 'S': 1, 'T': 1, 'U': 1, 'V': 4, 'W': 4, 'X': 8,
    'Y': 4, 'Z': 10
}

# Board multipliers for positions 1-13 (0-indexed: 0-12)
BOARD_MULTIPLIERS = [1, 1, 2, 1, 1, 1, 3, 1, 1, 1, 2, 1, 1]


def can_make_word(word, avail_counts):
    """Check if a word can be formed from available letters (no wildcard)."""
    word_counts = Counter(word)
    for letter, count in word_counts.items():
        if avail_counts.get(letter, 0) < count:
            return False
    return True


def can_make_word_wildcard(word, avail_counts):
    """Check if word can be formed from available letters + 1 wildcard."""
    word_counts = Counter(word)
    deficit = 0
    for letter, count in word_counts.items():
        shortfall = count - avail_counts.get(letter, 0)
        if shortfall > 0:
            deficit += shortfall
        if deficit > 1:
            return False
    return True


def find_valid_words(input_string, n, has_wildcard=False):
    """Find all valid words of length n from input_string."""
    if has_wildcard:
        letters = [c for c in input_string.upper() if c != '?']
    else:
        letters = list(input_string.upper())
    
    avail_counts = Counter(letters)
    candidates = words_by_length.get(n, [])
    valid = []
    
    if has_wildcard:
        for word in candidates:
            if can_make_word_wildcard(word, avail_counts):
                valid.append(word)
    else:
        for word in candidates:
            if can_make_word(word, avail_counts):
                valid.append(word)
    
    return sorted(set(valid))


def compute_word_score(word):
    """Compute the optimal board score for a word.
    The word runs left-to-right, position 7 (index 6) must be occupied."""
    word_len = len(word)
    best_score = 0
    min_start = max(0, 7 - word_len)
    max_start = min(6, 13 - word_len)
    
    for start in range(min_start, max_start + 1):
        score = 0
        for i, ch in enumerate(word):
            score += LETTER_POINTS[ch] * BOARD_MULTIPLIERS[start + i]
        if score > best_score:
            best_score = score
    
    return best_score


def find_highest_scoring_word(valid_words):
    """Find the highest scoring word (alphabetically first if tie)."""
    best_score = -1
    best_word = None
    
    for word in valid_words:
        score = compute_word_score(word)
        if score > best_score or (score == best_score and (best_word is None or word < best_word)):
            best_score = score
            best_word = word
    
    return best_word, best_score


# Verify with the example: FINANCE, N=6, X=3
example_words = find_valid_words('FINANCE', 6, has_wildcard=False)
print(f'Example FINANCE N=6: {len(example_words)} words')
print(f'Words: {example_words}')
if len(example_words) >= 3:
    print(f'3rd word: {example_words[2]}')
hw, hs = find_highest_scoring_word(example_words)
print(f'Highest: {hw} with score {hs}')

In [ ]:
# Cell 3: Parse question files and solve all 15 questions
import re

def parse_question_file(filepath):
    """Parse a question file to extract input string, N, X, and wildcard flag."""
    with open(filepath, 'r') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
    # Line 2 has: "1 ABILITY 5 1" or "11 ABACUS? 5 38"
    parts = lines[1].split()
    q_num = int(parts[0])
    input_str = parts[1].upper()
    n = int(parts[2])
    x = int(parts[3])
    has_wildcard = '?' in input_str
    return q_num, input_str, n, x, has_wildcard

# Find data directory
data_dir = None
for candidate in ['/workspace/data', 'data', '../data',
                  '/home/ddzhang/DSBench/colab_tasks/dsbench_da_00000009/environment/data']:
    if os.path.exists(os.path.join(candidate, 'question1.txt')):
        data_dir = candidate
        break

results = {}
for i in range(1, 16):
    qfile = os.path.join(data_dir, f'question{i}.txt')
    q_num, input_str, n, x, has_wc = parse_question_file(qfile)

    valid_words = find_valid_words(input_str, n, has_wildcard=has_wc)
    num_words = len(valid_words)
    xth_word = valid_words[x - 1] if x <= num_words else 'N/A'
    highest_word, highest_score = find_highest_scoring_word(valid_words)

    results[f'question{q_num}'] = {
        '# of words': num_words,
        'Xth word': xth_word,
        'Highest Word': highest_word,
        'Highest Score': highest_score
    }

    print(f'Q{q_num}: {input_str} N={n} X={x} | '
          f'#words={num_words}, Xth={xth_word}, '
          f'Best={highest_word} Score={highest_score}')

print(f'\nAll questions solved in {time.time()-start:.1f}s total.')

In [ ]:
# Cell 4: Set the final answers dict

answers = results

for key, val in answers.items():
    print(f'{key}: {val}')